# 06 — Principal Component Analysis

**Objective:** Plan PCA experiments and interpretation.  
**Owner:** Ilias El Hamri  
**Sprint:** 01  

> Leakage warning: fit scaling and PCA on training folds only.

## Project Setup

This cell locates the project root and loads the shared configuration.

- The project root is found by walking up from the current working directory until a folder containing `configs/config.yaml` is found. This ensures the notebook works regardless of where Jupyter is launched from.
- The project root is inserted into `sys.path` so that `src.*` imports resolve correctly.
- `load_config()` reads `configs/config.yaml` and returns a dictionary that contains shared settings such as `random_state`, output paths, and metric names.

In [ ]:
from pathlib import Path

project_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "configs" / "config.yaml").is_file())
import sys
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from src.config import load_config

config = load_config()
print(config["project"]["name"])
print(config["project"]["random_state"])

## Planned work

Later: evaluate explained variance and downstream performance using fold-local transformations and the shared validation protocol.

## Data Loading

This cell loads the Santander Customer Transaction dataset from OpenML using the shared utility.

- `optimize_memory=True` converts numeric features from `float64` to `float32`, reducing memory usage by roughly half while preserving all structural invariants.
- The function returns three objects:
  - `X` — a DataFrame of 200 anonymised feature columns.
  - `y` — a binary target Series (`"0"` / `"1"`).
  - `metadata` — a dictionary with dataset description and provenance information.

In [ ]:
from src.data import load_dataset

X, y, metadata = load_dataset(optimize_memory=True)

## Pipeline Construction

This cell builds the full PCA dimensionality-reduction pipeline using scikit-learn's `Pipeline` to prevent data leakage.

### Why scale before PCA?

- PCA is based on variance decomposition. Without scaling, features with larger numeric ranges dominate the principal components regardless of their predictive importance.
- `StandardScaler` normalises all 200 features to zero mean and unit variance before PCA is applied.

### PCA configuration

- `n_components=0.95` instructs PCA to automatically select the **minimum number of components** needed to preserve 95% of the total explained variance.
- This avoids guessing a fixed number of components and adapts to the data in each fold.
- `random_state` ensures reproducibility across runs.

### Final classifier

- A standard L2-regularised Logistic Regression trains on the compressed PCA components, not the original 200 features.

### Why a Pipeline?

- Bundling all three steps — scaling → PCA → classification — inside a `Pipeline` is mandatory.
- `evaluate_model_cv` refits the entire pipeline independently inside each cross-validation training fold, so `StandardScaler` and `PCA` are never exposed to validation or test data.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

pca = PCA(n_components=0.95, random_state=config["project"]["random_state"])

classifier = LogisticRegression(
    penalty="l2", solver="lbfgs", C=1.0, max_iter=1000, random_state=config["project"]["random_state"])

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", pca),
    ("classifier", classifier)
])

## Cross-Validation Evaluation

This cell evaluates the pipeline using the shared project infrastructure.

### What `evaluate_model_cv` does

- Applies 5-fold stratified cross-validation on **training data only**.
- The final test partition is never passed and remains entirely closed.
- Returns two objects:
  - `fold_results` — a DataFrame with per-fold metrics (ROC-AUC, Average Precision, F1, etc.).
  - `summary` — a dictionary with aggregate scores, estimator parameters, and experiment metadata.

### Parameters

- `experiment_id="M03-PCA-001"` — unique identifier for this experiment (Member 03, PCA, run 001).
- `member="Member 03"` — identifies Ilias El Hamri as the author.
- `branch="feature/pca"` — Git branch this notebook belongs to.

### Output

- The primary metric (ROC-AUC) mean and standard deviation across the five folds are printed. The standard deviation indicates how stable the model is across different data subsets.

In [ ]:
from src.evaluation import evaluate_model_cv

fold_results, summary = evaluate_model_cv(
    estimator=pipeline,
    X=X,
    y=y,
    model_name="PCA + LogisticRegression",
    experiment_id="M03-PCA-001",
    member="Member 03",
    branch="feature/pca"
)
print(f"Primary Metric ({summary['primary_metric']}): {summary['primary_score_mean']:.4f} +/- {summary['primary_score_std']:.4f}")